# ARC2 011 S0 shared model

Manual bounded research experiment, 2026-10-11. Fixed Qwen3.8-27B and verified S4 step25; no competition submission. Results below are produced by execution, not assumed. CPU preflight does not load model weights. GPU mode has a 120-minute notebook budget and a 20-minute model-load stop; it does not retry.

Revision 2 corrects the candidate event signature and resource-status precedence. Version 1 was cancelled before generation. CPU full-wrapper diagnostics must pass before this version is launched.


### 1. Restore frozen code and input-only task packets


In [ ]:
import base64, hashlib, io, json, pathlib, time, zipfile
NOTEBOOK_START=time.monotonic()
CODE=pathlib.Path('/kaggle/working/011/code'); CODE.mkdir(parents=True,exist_ok=True)
CODE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': '54b519deb5876f0f9f90008da69151b6ec4aa35b808b7c80d7d624f385c7d20f', 'p12-challenges.json': '792ca71c069abb420368efd1b0a29c04adaaed9761ba9225528db735cfe44a40', 'p12-queries.json': '884eb10df5a65bb527cd4c4c72edc7b5ce4dafd16f481e8fef8cc8478e7950f7', 'p12-scoring.json': 'fbe21ed91e38259e70aec5864994944759e2ec83dd0bf2c9093a7c549c562459', 'p6-queries.json': '02e7a837fddd9a46d4b8af2e0ef630acafd55655cfce10af73633be8134e2844', 'research-manifest.json': '301c10ad8768c232ba50a2b94cc9b3b4c975da9fdc3ea936b90a02719575ffeb', 'run_budgeted_grid_experiment.py': 'af5ae4f2baca3f1c670f9b50fc94b98dc46c04eaeea1cf55ec34ea0497107e23', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': 'bb500aef69a90b9613f74e0e33b44895f7d3b9771138481b500ba836605bb54d', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
EXPECTED_ADAPTER_SHA='9282c46cbce02923afbea3ad041614f0bc6ff3d7bd97c5174adfa3dd3e485ff9'
payload=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        if pathlib.PurePosixPath(name).name != name: raise ValueError('unsafe archive path')
        data=z.read(name)
        assert hashlib.sha256(data).hexdigest()==CODE_MANIFEST[name]
        (CODE/name).write_bytes(data)
(CODE/'source-manifest.json').write_text(json.dumps(CODE_MANIFEST,indent=2))
print('PACKAGED_INPUTS',len(CODE_MANIFEST))
CORE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': '54b519deb5876f0f9f90008da69151b6ec4aa35b808b7c80d7d624f385c7d20f', 'run_budgeted_grid_experiment.py': 'af5ae4f2baca3f1c670f9b50fc94b98dc46c04eaeea1cf55ec34ea0497107e23', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': 'bb500aef69a90b9613f74e0e33b44895f7d3b9771138481b500ba836605bb54d', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
CORE_MANIFEST_SHA='bbc9b83fc24f7b216878695cf7ca41fec83af15ab94bbb8af640047e2896a97e'


### 2. Prepare the existing offline environment


In [ ]:
import glob, os, subprocess, sys, time, json
wheel = glob.glob("/kaggle/input/**/qwen38-wheelhouse-py312", recursive=True)
if len(wheel) != 1:
    raise RuntimeError("Expected one frozen wheelhouse, found %d" % len(wheel))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, timeout=90)
p = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links=" + wheel[0], "transformers==5.8.0", "accelerate", "peft"],
                   capture_output=True, text=True, timeout=240)
print((p.stdout or "")[-1400:])
if p.returncode:
    print((p.stderr or "")[-1400:])
    raise RuntimeError("Offline environment preparation failed")
import transformers, torch, peft, accelerate
versions = dict(python=sys.version.split()[0], transformers=transformers.__version__,
                torch=torch.__version__, peft=peft.__version__, accelerate=accelerate.__version__)
print("VERSIONS", json.dumps(versions))
assert transformers.__version__ == "5.8.0"


### 3. Verify model and adapter identities


In [ ]:
from pathlib import Path
import hashlib
def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(2*1024*1024), b""):
            h.update(b)
    return h.hexdigest()
models = []
for p in Path("/kaggle/input").rglob("config.json"):
    try:
        c = json.loads(p.read_text())
    except Exception:
        continue
    if c.get("model_type") == "qwen3_5" and (p.parent/"model.safetensors.index.json").is_file():
        models.append(p.parent)
if len(models) != 1:
    raise RuntimeError("Expected one Qwen3.8 model directory, found %d" % len(models))
model_dir = str(models[0])
adapters = list(Path("/kaggle/input").rglob("adapter_step_25/adapter_model.safetensors"))
matching = [p.parent for p in adapters if digest(p) == EXPECTED_ADAPTER_SHA]
if len(matching) != 1:
    raise RuntimeError("Expected verified S4 adapter SHA, matching directories=%d" % len(matching))
adapter_dir = str(matching[0])
assert json.loads((matching[0]/"formatter.json").read_text())["formatter"] == "f0_space"
print("INPUT_IDENTITY", json.dumps(dict(model_dir=model_dir, adapter_dir=adapter_dir,
     adapter_sha256=EXPECTED_ADAPTER_SHA, model_config_sha256=digest(models[0]/"config.json"))))
sys.path.insert(0, str(CODE))


### 4. Freeze the two development stages

P12 F0-25 closes the previous comparison. P6 base F1 original/sample/transpose is a predeclared diagnostic. No confirmation tasks are loaded. Labels are available only to the post-generation scorer.


In [ ]:
config = {
 "model_dir": model_dir, "challenges": str(CODE/"p12-challenges.json"),
 "solutions": str(CODE/"p12-scoring.json"), "out_dir": "/kaggle/working/011/s0-shared",
 "time_budget_s": 7200, "load_limit_s": 1200,
 "notebook_start_monotonic": NOTEBOOK_START,
 "runtime": {"base_source":"kaggle:ugvfpdcuwfnh/qwen3-8-27b/Transformers/transformers-bf16/1",
             "deactivate_async_load":True, "save_window_s":300},
 "stages": [
  {"type":"grid","name":"s0_f0_25_p12","adapter":adapter_dir,"formatter":"f0_space",
   "query_file":str(CODE/"p12-queries.json"),"max_new_tokens":2048,
   "max_time_s":600,"estimate_s":240,
   "candidates":[{"view":"original","do_sample":False}]},
  {"type":"grid","name":"s1_base_f1_p6","adapter":None,"formatter":"f1_compact",
   "query_file":str(CODE/"p6-queries.json"),"max_new_tokens":960,
   "time_budget_s":1200,"max_time_s":300,"estimate_s":120,
   "candidates":[{"view":"original","do_sample":False},
                 {"view":"original","do_sample":True,"sample_index":1},
                 {"view":"transpose","do_sample":False}]}
 ]}
config_path=CODE/"run.json"
config_path.write_text(json.dumps(config,ensure_ascii=False,indent=2))
p=subprocess.run([sys.executable,str(CODE/"run_budgeted_grid_experiment.py"),
                  "--config",str(config_path),"--dry-run"],capture_output=True,text=True,timeout=60)
print(p.stdout)
if p.returncode:
    print(p.stderr[-1500:])
    raise RuntimeError("Configuration preflight failed")


### Match the successful CPU report to this exact core revision


In [ ]:

# The preflight dependency is matched on core code only, not its task payload.
# This deliberately lets the same CPU report support a later S0 P12 package.
reports=[]
for path in Path("/kaggle/input").rglob("cpu-pilot-preflight.json"):
    try:
        report=json.loads(path.read_text())
    except Exception:
        continue
    if (report.get("status")=="CPU_PILOT_PREFLIGHT_OK"
            and report.get("core_manifest")==CORE_MANIFEST
            and report.get("core_manifest_sha256")==CORE_MANIFEST_SHA):
        reports.append((path,report))
if len(reports)!=1:
    raise RuntimeError("Expected one successful CPU pilot preflight with the identical core SHA set")
path,report=reports[0]
for key in ("python","transformers","torch","peft","accelerate"):
    if versions[key]!=report["versions"][key]:
        raise RuntimeError("CPU/GPU environment differs for "+key)
print("MATCHED_CPU_PREFLIGHT",json.dumps({"path":str(path),"core_manifest_sha256":CORE_MANIFEST_SHA,
      "payload_equality_required":False,"pretrained_weights_checked_by_cpu":False}))


### 5. Run within the hard budget


In [ ]:
assert torch.cuda.device_count()==4,"Expected four L4 devices"
print("HARD_LIMITS",json.dumps({"notebook_seconds":7200,"load_seconds":1200,
      "already_elapsed_s":time.monotonic()-NOTEBOOK_START}))
remaining=7200-(time.monotonic()-NOTEBOOK_START)
if remaining < 900:
    raise RuntimeError("Insufficient budget after setup; refusing model load")
out=Path(config["out_dir"]);out.mkdir(parents=True,exist_ok=True)
telemetry_file=(out/"gpu-telemetry.csv").open("w")
telemetry=None
try:
    telemetry=subprocess.Popen(["nvidia-smi",
        "--query-gpu=timestamp,index,name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw",
        "--format=csv", "-l", "30"],stdout=telemetry_file,stderr=subprocess.DEVNULL)
    proc=subprocess.run([sys.executable,str(CODE/"run_budgeted_grid_experiment.py"),
                         "--config",str(config_path)],timeout=max(1,remaining+15))
finally:
    if telemetry is not None:
        telemetry.terminate()
        try:
            telemetry.wait(timeout=10)
        except subprocess.TimeoutExpired:
            telemetry.kill();telemetry.wait()
    telemetry_file.close()
print("EXPERIMENT_EXIT",proc.returncode)
for name in ["supervisor.json","SUMMARY.json"]:
    path=out/name
    if path.is_file():
        print(name,path.read_text()[:8000])
if proc.returncode not in (0,2):
    raise RuntimeError("Experiment infrastructure error, inspect worker.log")


### Interpretation

Read the generated JSON and worker journal. A successful CPU diagnostic establishes preparation only. GPU outcomes must include all requested queries, explicit incomplete cases, and actual timing. No result here is a competition score.
